# Sparse-Oracle Block Encoding and the FABLE Algorithm

*Block encoding* is the workhorse primitive behind modern quantum linear algebra. It is commonly applied in
Hamiltonian simulation and the quantum singular value transformation (QSVT), Monte-Carlo estimation and utilized by linear-system
solvers. This notebook is a
technical, algorithm-focused tour of the **sparse-oracle** block-encoding methods in the
Classiq SDK, followed by a description of the **FABLE** algorithm for *dense* matrices.

A unitary $U$ is an $(\alpha, m, \epsilon)$-block-encoding of a matrix $A$ (acting on
$n$ data qubits) if

$$
\left\| A - \alpha\,\bigl(\langle 0|^{\otimes m} \otimes I_{2^n}\bigr)\,U\,\bigl(|0\rangle^{\otimes m} \otimes I_{2^n}\bigr) \right\| \le \epsilon .
$$

In words, $A/\alpha$ sits in the top-left block of $U$, addressed by setting the $m$
block (ancilla) qubits to $|0\rangle$. Here $\alpha$ is the **subnormalization factor**
(needed because $U$ is unitary, so $\|A/\alpha\| \le 1$), $m$ is the **block size**, and
$\epsilon$ is the encoding error.


In the following notebook we will:

1. Define the sparse-oracle framework and build an encoding from hand-written oracles.
2. Present three circuit variants to create a sparse block encoding (`lcu`, `general`, `hermitian`).
3. Demonstrate how to use the higher-level builders `from_scipy_sparse` and `from_sparsity_pattern`.
4. Compose encodings and form a qubitization walk operator.
5. Describes the FABLE dense-matrix algorithm and its implementation.

## 1. The sparse-oracle contract

An $s$-sparse matrix has at most $s$ nonzero entries in every row and column. Instead
of writing down all $2^{2n}$ amplitudes, we access $A$ through a small number of oracles
that answer local questions, this is what makes sparse access exponentially cheaper than
a dense description [[3](#BACS07)].

For a matrix of dimension $N = 2^n$ we supply (writing $|\cdot\rangle_{\text{idx}}$ for the
sparse-index quantum variable, $|\cdot\rangle_{\text{d}}$ for the data quantum variable, and
$|\cdot\rangle_{\text{a}}$ for the amplitude ancilla):

- **Column oracle $O_c$** — `oracle_column(index, data)`: given the sparse index
  $k \in \{0,\dots,s-1\}$ and a row $i$ (held in `data`), it writes the column
  $c(i, k)$ of the $k$-th nonzero of row $i$ into `data`:
  $$
  O_c\;|k\rangle_{\text{idx}}\,|i\rangle_{\text{d}}
  \;=\; |k\rangle_{\text{idx}}\,|c(i,k)\rangle_{\text{d}}.
  $$
- **Amplitude oracle $O_A$** — `oracle_amplitude(amp, index, data)`: rotates the single
  ancilla qubit `amp` so its $|0\rangle$-amplitude equals the (normalized) entry
  $A_{i,\,c(i,k)}$, via an $R_y$ rotation with angle $\theta_{ik} = 2\arccos A_{i,c(i,k)}$:
  $$
  O_A\;|0\rangle_{\text{a}}\,|k\rangle_{\text{idx}}\,|i\rangle_{\text{d}}
  \;=\; \Bigl(A_{i,c(i,k)}\,|0\rangle + \sqrt{1-|A_{i,c(i,k)}|^{2}}\;|1\rangle\Bigr)_{\text{a}}
  \,|k\rangle_{\text{idx}}\,|i\rangle_{\text{d}} .
  $$
- **Row oracle $O_r$** — `oracle_row(index, data)`: the transpose of $O_c$, mapping a
  column $j$ to the row $r(j,k)$ of its $k$-th nonzero:
  $$
  O_r\;|k\rangle_{\text{idx}}\,|j\rangle_{\text{d}}
  \;=\; |k\rangle_{\text{idx}}\,|r(j,k)\rangle_{\text{d}} .
  $$

The implementation imposes to hard constraints on the input:

- `sparsity` must be a positive power of two (the sparse index quantum variable is
  $\lceil \log_2 s\rceil$ qubits).
- `data_size` ($=n$) is passed explicitly — the oracles are generic quantum
  subroutines, so the matrix dimension cannot be inferred from them.

The raw entry point is `BlockEncoding.from_sparse_oracles`, whose subnormalization is
exactly $\alpha = s$.

## 2. Setup

We begin by importing the `BlockEncoding` class and sparsity patterns. The helper below compares a reconstructed matrix
to the expected one up to a global phase, as `to_matrix()` block encoding method recovers $A$ only up to an
overall phase, which is physically unobservable.

In [ ]:
import numpy as np
import scipy.sparse as sp

from classiq import *
from classiq.applications.block_encoding import BlockEncoding
from classiq.applications.block_encoding.sparse import (
    CirculantPattern,
    TridiagonalPattern,
    from_scipy_sparse,
)


def assert_close_up_to_global_phase(actual, expected, atol=1e-5):
    # Assert two matrices are equal up to a single (unknown) global phase.
    actual = np.asarray(actual, dtype=complex)
    expected = np.asarray(expected, dtype=complex)
    overlap = np.vdot(expected, actual)
    assert abs(overlap) > atol, "matrices are (nearly) orthogonal; phase is undefined"
    phase = overlap / abs(overlap)
    np.testing.assert_allclose(actual, phase * expected, atol=atol)
    print("reconstruction confirmed")

## 3. A worked example from hand-written oracles

We begin with a simple block encoding applying the `lcu` algorithm variant (see Sec. 4).

Consider the $2\times2$ symmetric matrix

$$
A = \begin{pmatrix} a_1 & a_2 \\ a_2 & a_1 \end{pmatrix},
\qquad a_1 = 1.0,\ a_2 = 0.5 .
$$

Every row has $s = 2$ nonzeros, and $n = 1$ data qubit. The oracles are given by:

- $O_c$ is a single `CX(index[0], data[0])`: sparse index $0$ keeps the column
  ($c=i$, the diagonal), sparse index $1$ flips it ($c = i \oplus 1$, the off-diagonal).
- $O_A$ loads $a_1$ with a base rotation `RY(2 arccos(a1))`, and conditioned on
  `index[0]` adds the delta `RY(2 arccos(a2) - 2 arccos(a1))` so that index $1$ loads
  $a_2$ instead.

In [ ]:
a1, a2 = 1.0, 0.5
theta1 = 2 * np.arccos(a1)
theta2 = 2 * np.arccos(a2)


@qfunc
def oracle_amplitude(amp: QBit, index: QArray[QBit], data: QArray[QBit]) -> None:
    RY(theta1, amp)
    control(index[0], lambda: RY(theta2 - theta1, amp))


@qfunc
def oracle_column(index: QArray[QBit], data: QArray[QBit]) -> None:
    CX(index[0], data[0])


be = BlockEncoding.from_sparse_oracles(
    oracle_amplitude=oracle_amplitude,
    oracle_column=oracle_column,
    sparsity=2,
    data_size=1,
)
print(f"alpha = {be.alpha},  block_size = {be.block_size},  data_size = {be.data_size}")

expected = np.array([[a1, a2], [a2, a1]])
assert_close_up_to_global_phase(be.to_matrix(), expected)

`be.unitary` is a Qmod function of `(data, block)`. To inspect the circuit we wrap it in a
`main` that allocates both quantum variables and synthesize it.

In [ ]:
@qfunc
def main(data: Output[QNum], block: Output[QNum]) -> None:
    allocate(be.data_size, data)
    allocate(be.block_size, block)
    be.unitary(data, block)


qprog = synthesize(main)
show(qprog)

## 4. The three circuit variants

`from_sparse_oracles` supports three constructions, selected with `variant=`. All three
share $\alpha = s$ but differ in ancilla count and which oracles they need.

| Variant       | Block size          | Oracles needed | Best for                                  |
| ------------- | ------------------- | -------------- | ----------------------------------------- |
| **`lcu`**     | $1 + \log_2 s$      | $O_c,\,O_A$    | very sparse ($s \ll N$), regular patterns |
| **`general`** | $1 + n$             | $O_r,O_c,O_A$  | non-Hermitian / irregular sparsity        |
| **`hermitian`** | $2 + n$           | $O_c,\,O_A$    | Hermitian matrices, qubitization          |

- **`lcu`** decomposes an $s$-sparse matrix into $s$ 1-sparse matrices (Kőnig's edge
  colouring) and superposes them with a PREPARE mapping on the index quantum variable. The
  core algorithm is $U_{\text{lcu}} = H\,(O_A O_c)\,H$.
- **`general`** uses row *and* column oracles plus a `SWAP` between data and index quantum variables. The applied unitary is:
  $H\,(O_c\,O_A\,\mathrm{SWAP}\,O_r^\dagger)\,H$. This is the generic construction of
  [[1](#GSLW19)].
- **`hermitian`** exploits $A_{ij}=A_{ji}^{*}$ so it requires only the column oracle (the row oracle is identical to the column oracle). A
  Hadamard-controlled qubit symmetrizes the encoding, yielding a *Hermitian* $U$ that is
  ready for qubitization [[2](#CKS17)].

Below we encode the same symmetric matrix with all three (the `general` variant needs the
extra row oracle, which for this symmetric case equals the column oracle).

In [ ]:
a1, a2 = 0.75, 0.25
theta1 = 2 * np.arccos(a1)
theta2 = 2 * np.arccos(a2)


@qfunc
def oracle_amplitude_v(amp: QBit, index: QArray[QBit], data: QArray[QBit]) -> None:
    RY(theta1, amp)
    control(index[0], lambda: RY(theta2 - theta1, amp))


@qfunc
def oracle_column_v(index: QArray[QBit], data: QArray[QBit]) -> None:
    CX(index[0], data[0])


@qfunc
def oracle_row_v(index: QArray[QBit], data: QArray[QBit]) -> None:
    CX(index[0], data[0])


expected = np.array([[a1, a2], [a2, a1]])

for variant in ["lcu", "general", "hermitian"]:
    be_v = BlockEncoding.from_sparse_oracles(
        oracle_amplitude=oracle_amplitude_v,
        oracle_column=oracle_column_v,
        sparsity=2,
        data_size=1,
        oracle_row=oracle_row_v if variant == "general" else None,
        variant=variant,
    )
    print(f"variant={variant:10s} block_size={be_v.block_size}  alpha={be_v.alpha}")
    assert_close_up_to_global_phase(be_v.to_matrix(), expected)

## 5. Higher-level builders

Writing oracles by hand is instructive but tedious. Two convenience constructors build the
oracles for you.

### 5.1 From an explicit matrix: `from_scipy_sparse`

Given any `scipy.sparse` matrix (square, power-of-two dimension), `from_scipy_sparse`
extracts the sparsity structure, detects Hermiticity, and builds **QROM-based** oracles
(gray-code multiplexed `select_rotation`). The subnormalization it targets is
$\alpha = s\cdot\max_{ij}|A_{ij}|$, and `variant="auto"` picks `hermitian` for Hermitian
inputs, otherwise `lcu`/`general` by ancilla economy.

> **Note.** These convenience builders normalize the amplitudes internally, so the
> reconstructed matrix carries the $\max|A_{ij}|$ scaling in $\alpha$. Below we only
> construct the encoding and read off its metadata; see the *Technical Notes* for the
> subnormalization bookkeeping.

In [ ]:
dense = np.array(
    [
        [2.0, 1.0, 0.0, 1.0],
        [1.0, 2.0, 1.0, 0.0],
        [0.0, 1.0, 2.0, 1.0],
        [1.0, 0.0, 1.0, 2.0],
    ]
)  # a symmetric, circulant-like 4x4 (Hermitian -> variant "auto" picks "hermitian")

mat = sp.csr_matrix(dense)
be_scipy = BlockEncoding.from_scipy_sparse(mat, variant="auto")
print(
    f"data_size = {be_scipy.data_size}, "
    f"block_size = {be_scipy.block_size}, "
    f"alpha = {be_scipy.alpha}, "
    f"hermitian_be = {be_scipy.hermitian_be}"
)

### 5.2 From a structured pattern

When the matrix has known structure, `from_sparsity_pattern` uses **arithmetic** oracles
(modular addition, controlled shifts) instead of QROM lookups. As a consequence it is asymptotically cheaper
because there is no table to load. The available `SparsityPattern` subclasses are:

- `BandedPattern` / `TridiagonalPattern` — nonzeros within $|i-j| \le b$; the column
  oracle is a single `inplace_add(index - b, data)`.
- `CirculantPattern` — $A_{ij}=f\!\bigl((j-i)\bmod N\bigr)$, from the first row.
- `LatticePattern` — $d$-dimensional discrete Laplacians (periodic or open).
- `TransverseFieldIsingPattern`, `HeisenbergXXZPattern` — physics Hamiltonians in the
  computational basis (diagonal $ZZ$ + off-diagonal spin flips).
- `GraphLaplacianPattern` — $L = D - A$ for an arbitrary edge list.

Each pattern also takes `impl="qrom"` (gray-code `select_rotation`, $O(L\log L)$ T-count)
or `impl="unary"` (unary iteration, $O(L)$ T-count, better for fault-tolerant regimes)
for the amplitude/column oracles. We construct a tridiagonal and a circulant encoding and
read off their metadata.

In [ ]:
# Tridiagonal (2 on the diagonal, -1 off-diagonal) on 2 data qubits (4x4).
tridiag = TridiagonalPattern(diag=2.0, off_diag=-1.0)
be_tri = BlockEncoding.from_sparsity_pattern(tridiag, data_size=2, variant="auto")
print(
    f"tridiagonal : data_size={be_tri.data_size} "
    f"block_size={be_tri.block_size} alpha={be_tri.alpha}"
)

# Circulant defined by its first row.
circ = CirculantPattern(first_row=np.array([2.0, 1.0, 0.0, 1.0]))
be_circ = BlockEncoding.from_sparsity_pattern(circ, data_size=2, variant="auto")
print(
    f"circulant   : data_size={be_circ.data_size} "
    f"block_size={be_circ.block_size} alpha={be_circ.alpha}"
)

## 6. Composing and transforming block encodings

The `BlockEncoding` objects built above are not dead ends: block encodings form an algebra,
so they can be combined and transformed without ever leaving the block-encoding formalism.
A `BlockEncoding` supports, among others:

- **Arithmetic** — sums and linear combinations via LCU (`+`, `BlockEncoding.weighted_sum`),
  matrix products (`@` / `BlockEncoding.product`), and scalar multiplication (`c * be`).
- **Inversion** — a block encoding of $A^{-1}$ via QSVT polynomial inversion
  (`be.inverse(kappa, degree=...)`).
- **Qubitization** — the walk operator $W = R\,U$ whose powers yield Chebyshev polynomials
  of $A/\alpha$ (`be.qubitize()`), the engine behind Hamiltonian simulation and QSVT.
  This requires the *unitary* to be Hermitian (`hermitian_be=True`), which the `hermitian`
  variant provides.

Each of these propagates `alpha`, `block_size`, and `hermitian_be` automatically, so sparse
and dense encodings interoperate freely.

## 7. FABLE: fast block encoding of *dense* matrices

The sparse oracles above shine when $A$ has few nonzeros. For a dense matrix an alternative strategy is preferable. In the following section we introduce the **FABLE** (Fast Approximate BLock Encodings) algorithm, [[4](#FABLE)], which writes down every entry directly as a uniformly-controlled rotation, then compresses
the circuit by discarding small rotation angles (in the Hadamard space).

> The FABLE constructor (`BlockEncoding.from_matrix(mat, method="fable", ...)`) is being
> added on a separate development branch and is not part of the environment this notebook
> runs in, so the FABLE cells below are shown for reference and are **not executed** here.

**Key quantities** for an $N\times N$ matrix ($N = 2^n$, non-power-of-two inputs are
zero-padded):

- subnormalization $\alpha = \max_{ij}|A_{ij}|\cdot N$;
- block size $n+1$ (an $n$-qubit column quantum variable plus one amplitude ancilla),
  data size $n$;
- error $\epsilon = N^3 \cdot \texttt{angle\_threshold}$ (a loose upper bound; exact when
  the threshold is $0$).

**Algorithm.** FABLE builds the circuit $H \to \mathrm{UCR} \to \mathrm{SWAP} \to H$:

1. Encode each entry as a rotation angle. Real matrices need only $R_y$ angles
   $\theta_{ij} = 2\arccos(A_{ij}/s)$ (the sign is carried by $\arccos$). Complex matrices
   add an $R_z$ pass with phase angles $-2\arg(A_{ij})$.
2. Transform the flat angle vector with a **fast Walsh–Hadamard transform** and reorder it
   into **Gray-code** order — $O(N\log N)$ instead of the naive $O(N^2)$.
3. Apply the angles as a **uniformly-controlled rotation**: between consecutive rotations a
   single CNOT (chosen by the Gray-code difference) walks the control quantum variable through all
   states, so each rotation is applied exactly once.

**Compression.** Passing `angle_threshold > 0` drops rotations whose (transformed) angle is
below the threshold. Crucially, the magnitude ($R_y$) and phase ($R_z$) passes are
compressed *independently*, and the phase mask is *gated by* the magnitude mask: a phase is
only meaningful on a nonzero magnitude, so if a position's magnitude is dropped its phase is
dropped too (but not vice versa). When a rotation is removed, the surrounding CNOTs cancel
pairwise ($\mathrm{CX}^2 = I$), keeping a single CNOT per control with odd parity so the
Gray-code traversal — and hence the encoded unitary — is preserved.

```python
# For reference only -- requires the FABLE branch.
import numpy as np
from classiq.applications.block_encoding import BlockEncoding

A = np.array([[0.5, 0.3, 0.1, 0.2],
              [0.4, 0.6, 0.2, 0.3],
              [0.2, 0.1, 0.7, 0.4],
              [0.1, 0.2, 0.3, 0.5]])

# Exact dense block encoding: alpha = max|A_ij| * 4, block_size = 3, data_size = 2.
be = BlockEncoding.from_matrix(A, method="fable")

# Approximate (compressed) encoding: drop small rotation angles for a shallower circuit.
be_approx = BlockEncoding.from_matrix(A, method="fable", angle_threshold=0.1)
```

## Technical Notes

**Subnormalization factors differ by constructor.**

- `from_sparse_oracles` hard-codes $\alpha = s$ (the sparsity). The amplitude oracle is
  expected to load the entries directly, so `to_matrix()` returns $A$ exactly (up to global
  phase), as verified above.
- `from_scipy_sparse` / `from_sparsity_pattern` *normalize* the amplitudes by
  $\max_{ij}|A_{ij}|$ before building QROM/arithmetic oracles, targeting
  $\alpha = s\cdot\max|A_{ij}|$. When comparing a reconstruction to the original matrix,
  account for this scaling (it coincides with the raw case only when $\max|A_{ij}| = 1$).
- FABLE uses $\alpha = \max_{ij}|A_{ij}|\cdot N$.

**Power-of-two requirements.** The sparse index quantum variable is $\lceil\log_2 s\rceil$ qubits,
so `sparsity` must be a power of two The
builders round the raw sparsity up to the next power of two automatically.

**`hermitian_be` propagation.** `hermitian_be` records whether the *unitary* $U$ is
Hermitian (required by `qubitize`), which is stronger than $A$ being Hermitian. Combinators
propagate it conservatively (e.g. a sum is Hermitian only if both summands are), but the
sparse-oracle constructors leave it at its default `False`; set it explicitly with
`dataclasses.replace` when you know $U$ is Hermitian (for instance before calling
`qubitize()`, which requires it).

## References

<a id='GSLW19'>[1]</a>: [A. Gilyén, Y. Su, G. H. Low, N. Wiebe, *Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics*, STOC 2019 (arXiv:1806.01838)](https://arxiv.org/abs/1806.01838)

<a id='CKS17'>[2]</a>: [A. M. Childs, R. Kothari, R. D. Somma, *Quantum algorithm for systems of linear equations with exponentially improved dependence on precision*, SIAM J. Comput. 46 (2017) (arXiv:1511.02306)](https://arxiv.org/abs/1511.02306)

<a id='BACS07'>[3]</a>: [D. W. Berry, G. Ahokas, R. Cleve, B. C. Sanders, *Efficient quantum algorithms for simulating sparse Hamiltonians*, Commun. Math. Phys. 270 (2007) (arXiv:quant-ph/0508139)](https://arxiv.org/abs/quant-ph/0508139)

<a id='FABLE'>[4]</a>: [D. Camps, R. Van Beeumen, *FABLE: Fast Approximate Quantum Circuits for Block-Encodings*, IEEE QCE 2022 (arXiv:2205.00081)](https://arxiv.org/abs/2205.00081)

<a id='LowChuang'>[5]</a>: [G. H. Low, I. L. Chuang, *Hamiltonian simulation by qubitization*, Quantum 3, 163 (2019) (arXiv:1610.06546)](https://arxiv.org/abs/1610.06546)